# 使用PyTorch构建图像分类MLP

> 本笔记本是 [Keras Sequential API版本](./02-顺序API构建分类MLP.ipynb) 的PyTorch等价实现，使用相同的Fashion-MNIST数据集和MLP架构，帮助你对比理解两大深度学习框架。

## 学习目标

1. 理解图像分类任务的数据预处理流程
2. 掌握多分类MLP的模型架构设计
3. 学会使用交叉熵损失函数
4. 理解Softmax输出和概率预测
5. 对比TensorFlow/Keras与PyTorch的核心API差异

## 数据集介绍

Fashion-MNIST数据集包含10类时尚商品的灰度图像：
- 60,000张训练图像
- 10,000张测试图像
- 图像尺寸：28×28像素
- 10个类别：T恤、裤子、套头衫、连衣裙、外套、凉鞋、衬衫、运动鞋、包、短靴

## 1. 环境配置

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# 设置随机种子确保结果可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# 自动选择设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch版本: {torch.__version__}")
print(f"torchvision版本: {torchvision.__version__}")
print(f"使用设备: {device}")

## 2. 数据加载与探索

In [ ]:
# 使用torchvision加载Fashion-MNIST数据集
# 定义类别名称
class_names = [
    'T-shirt/top',  # T恤
    'Trouser',      # 裤子
    'Pullover',     # 套头衫
    'Dress',        # 连衣裙
    'Coat',         # 外套
    'Sandal',       # 凉鞋
    'Shirt',        # 衬衫
    'Sneaker',      # 运动鞋
    'Bag',          # 包
    'Ankle boot'    # 短靴
]

# 先加载原始数据（仅ToTensor，不做归一化）用于探索
raw_transform = transforms.ToTensor()
raw_train_set = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=raw_transform
)
raw_test_set = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=raw_transform
)

# 查看数据集基本信息
print("数据集大小:")
print(f"训练集: {len(raw_train_set)} 样本")
print(f"测试集: {len(raw_test_set)} 样本")

# 查看单个样本
sample_img, sample_label = raw_train_set[0]
print(f"\n单张图像形状: {sample_img.shape}")  # [1, 28, 28] - (C, H, W)
print(f"数据类型: {sample_img.dtype}")
print(f"像素值范围: [{sample_img.min():.1f}, {sample_img.max():.1f}]")

In [ ]:
# 查看标签分布
print("训练集标签分布:")
all_labels = [raw_train_set[i][1] for i in range(len(raw_train_set))]
unique, counts = np.unique(all_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {class_names[label]}: {count}")

In [ ]:
# 可视化部分样本
def plot_samples(dataset, class_names, n_rows=3, n_cols=5):
    """
    绘制数据集样本
    Plot dataset samples.

    Parameters:
    -----------
    dataset : torch Dataset
        图像数据集
    class_names : list
        类别名称列表
    n_rows, n_cols : int
        行数和列数
    """
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 7))
    for i, ax in enumerate(axes.flat):
        img, label = dataset[i]
        # PyTorch图像格式: (C, H, W) -> matplotlib需要 (H, W, C) 或 (H, W)
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(class_names[label], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_samples(raw_train_set, class_names)

## 3. 数据预处理

### 关键步骤

1. **划分验证集**: 从训练集中分离出验证数据
2. **归一化**: 使用torchvision.transforms将像素值从[0, 255]归一化到[-1, 1]或[0, 1]
3. **展平**: MLP需要一维输入（nn.Flatten层会在模型中处理）

### PyTorch vs Keras数据预处理差异

- **Keras**: 手动除以255.0归一化，使用NumPy数组
- **PyTorch**: 使用transforms管道归一化，使用Dataset + DataLoader
- **PyTorch图像格式**: 通道在前 (C, H, W)，Keras为 (H, W, C)

In [ ]:
# 使用torchvision.transforms进行归一化
# ToTensor() 将PIL图像转为[0,1]范围的Tensor (C, H, W)
# Normalize() 进一步标准化：这里使用均值0.5, 标准差0.5将[0,1]映射到[-1,1]
# 也可以只用ToTensor()保持[0,1]范围，与Keras版本一致

transform = transforms.Compose([
    transforms.ToTensor(),  # 将PIL Image转为Tensor，自动归一化到[0, 1]
    # transforms.Normalize((0.5,), (0.5,))  # 可选：映射到[-1, 1]
])

# 加载完整训练集和测试集
full_train_set = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_set = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# 划分验证集（前5000个样本作为验证集，与Keras版本一致）
valid_size = 5000
train_size = len(full_train_set) - valid_size

# 使用random_split划分（使用固定种子确保可复现）
generator = torch.Generator().manual_seed(RANDOM_SEED)
train_set, valid_set = random_split(full_train_set, [train_size, valid_size], generator=generator)

print(f"训练集大小: {len(train_set)}")
print(f"验证集大小: {len(valid_set)}")
print(f"测试集大小: {len(test_set)}")

# 创建DataLoader
BATCH_SIZE = 32

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

# 验证归一化后的数据范围
sample_batch_X, sample_batch_y = next(iter(train_loader))
print(f"\n批次数据形状: {sample_batch_X.shape}")  # (32, 1, 28, 28)
print(f"批次标签形状: {sample_batch_y.shape}")    # (32,)
print(f"归一化后像素值范围: [{sample_batch_X.min():.1f}, {sample_batch_X.max():.1f}]")

## 4. 构建MLP分类模型

### 模型架构

```
输入层 (1×28×28 = 784个像素)
    ↓ Flatten
展平层 (784个神经元)
    ↓
隐藏层1 (300个神经元, ReLU)
    ↓
隐藏层2 (100个神经元, ReLU)
    ↓
输出层 (10个神经元, 原始logits)
```

### 设计要点

- **nn.Flatten层**: 将(C, H, W)图像展平为一维向量
- **ReLU激活**: 隐藏层使用ReLU解决梯度消失问题
- **输出层无激活**: PyTorch的CrossEntropyLoss内部包含Softmax，因此输出层直接输出原始logits

> **重要区别**: Keras版本输出层使用Softmax激活，而PyTorch版本输出原始logits，因为`nn.CrossEntropyLoss` = `LogSoftmax` + `NLLLoss`，数值更稳定。

In [ ]:
# 使用nn.Sequential构建模型（等价于Keras的Sequential API）
model = nn.Sequential(
    # 输入层：Flatten将(1, 28, 28)图像展平为784维向量
    # start_dim=1 保留batch维度，从第1维开始展平
    nn.Flatten(start_dim=1),

    # 隐藏层1：300个神经元，ReLU激活
    # 等价于 keras.layers.Dense(300, activation='relu')
    nn.Linear(28 * 28, 300),
    nn.ReLU(),

    # 隐藏层2：100个神经元，ReLU激活
    # 等价于 keras.layers.Dense(100, activation='relu')
    nn.Linear(300, 100),
    nn.ReLU(),

    # 输出层：10个神经元（对应10个类别），无激活函数
    # 等价于 keras.layers.Dense(10, activation='softmax') 但不包含softmax
    # CrossEntropyLoss内部会自动应用Softmax
    nn.Linear(100, 10)
)

# 将模型移到设备上
model = model.to(device)

# 查看模型结构
print(model)

In [ ]:
# 查看模型参数详情
print("模型层信息:")
print("=" * 60)
total_params = 0
for name, param in model.named_parameters():
    num_params = param.numel()
    total_params += num_params
    print(f"{name}: 形状={param.shape}, 参数数={num_params}")
print(f"\n总参数数: {total_params}")

# 对比Keras的model.summary()输出
print("\n等价于Keras model.summary():")
print("-" * 60)
print(f"{'Layer (type)':<30} {'Output Shape':<20} {'Param #':<10}")
print("=" * 60)
dummy_input = torch.randn(1, 1, 28, 28).to(device)
for name, module in model.named_children():
    dummy_input_out = module(dummy_input)
    params_in_layer = sum(p.numel() for p in module.parameters())
    print(f"{name:<30} {str(list(dummy_input_out.shape)):<20} {params_in_layer:<10}")
    dummy_input = dummy_input_out
print("=" * 60)
print(f"Total params: {total_params}")

## 5. 损失函数与优化器

### 损失函数选择

- **Keras**: `sparse_categorical_crossentropy` — 标签为整数形式 (0, 1, 2, ...)
- **PyTorch**: `nn.CrossEntropyLoss` — 同样接受整数标签，无需one-hot编码

### 关键区别

| 特性 | Keras | PyTorch |
|------|-------|--------|
| 损失函数 | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss` |
| 输入要求 | Softmax概率 + 整数标签 | 原始logits + 整数标签 |
| 内部Softmax | 否（需手动） | 是（内置LogSoftmax） |
| 数值稳定性 | 一般 | 更好（log-sum-exp技巧） |

In [ ]:
# 定义损失函数和优化器
# 等价于 model.compile(loss='sparse_categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

criterion = nn.CrossEntropyLoss()  # 包含Softmax，等价于sparse_categorical_crossentropy
optimizer = optim.SGD(model.parameters(), lr=0.01)  # SGD优化器，学习率0.01

print(f"损失函数: {criterion}")
print(f"优化器: {optimizer}")
print(f"学习率: {optimizer.param_groups[0]['lr']}")

## 6. 训练模型

### PyTorch训练循环 vs Keras的model.fit()

Keras使用高层API `model.fit()` 一行代码完成训练，PyTorch需要手动编写训练循环，这提供了更大的灵活性：

1. **前向传播**: `outputs = model(X_batch)`
2. **计算损失**: `loss = criterion(outputs, y_batch)`
3. **反向传播**: `loss.backward()`
4. **更新参数**: `optimizer.step()`
5. **清零梯度**: `optimizer.zero_grad()`

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    训练模型一个epoch。
    Train the model for one epoch.

    Parameters:
    -----------
    model : nn.Module
        要训练的模型
    dataloader : DataLoader
        训练数据加载器
    criterion : nn.Module
        损失函数
    optimizer : optim.Optimizer
        优化器
    device : torch.device
        计算设备

    Returns:
    --------
    tuple : (平均损失, 准确率)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / len(dataloader.dataset), correct / total


def eval_epoch(model, dataloader, criterion, device):
    """
    在验证/测试集上评估模型。
    Evaluate the model on validation/test set.

    Parameters:
    -----------
    model : nn.Module
        要评估的模型
    dataloader : DataLoader
        验证/测试数据加载器
    criterion : nn.Module
        损失函数
    device : torch.device
        计算设备

    Returns:
    --------
    tuple : (平均损失, 准确率)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / len(dataloader.dataset), correct / total

In [ ]:
# 训练模型
# 等价于 Keras的 model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_valid, y_valid))

NUM_EPOCHS = 30

# 记录训练历史
history = {
    'loss': [], 'accuracy': [],
    'val_loss': [], 'val_accuracy': []
}

print("开始训练...")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
print("-" * 56)

for epoch in range(NUM_EPOCHS):
    # 训练一个epoch
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    # 验证
    val_loss, val_acc = eval_epoch(model, valid_loader, criterion, device)

    # 记录历史
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    # 打印进度
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:<8} {train_loss:<12.4f} {train_acc:<12.4f} {val_loss:<12.4f} {val_acc:<12.4f}")

print("\n训练完成!")

## 7. 可视化训练过程

In [ ]:
# 绘制学习曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 损失曲线
axes[0].plot(history['loss'], label='Training Loss')
axes[0].plot(history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(history['accuracy'], label='Training Accuracy')
axes[1].plot(history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 使用pandas绘制所有指标
history_df = pd.DataFrame(history)
history_df.plot(figsize=(10, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('Training History')
plt.show()

## 8. 模型评估

In [ ]:
# 在测试集上评估模型
# 等价于 Keras的 model.evaluate(X_test, y_test)
test_loss, test_accuracy = eval_epoch(model, test_loader, criterion, device)

print("测试集评估结果:")
print(f"损失 (Cross-Entropy): {test_loss:.4f}")
print(f"准确率: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 9. 使用模型进行预测

In [ ]:
# 对测试集前3个样本进行预测
# 等价于 Keras的 model.predict(X_new)

model.eval()
X_new_list = []
y_true_list = []
for i in range(3):
    img, label = test_set[i]
    X_new_list.append(img)
    y_true_list.append(label)

X_new = torch.stack(X_new_list).to(device)
y_true = np.array(y_true_list)

with torch.no_grad():
    logits = model(X_new)  # 输出原始logits
    # 手动应用Softmax获取概率（等价于Keras的predict输出）
    y_proba = torch.softmax(logits, dim=1).cpu().numpy()

print("预测概率分布:")
print(y_proba.round(3))

In [ ]:
# 获取预测类别（概率最高的类别）
y_pred = np.argmax(y_proba, axis=1)

print("预测结果:")
print("=" * 50)
for i in range(len(X_new_list)):
    true_label = class_names[y_true[i]]
    pred_label = class_names[y_pred[i]]
    confidence = y_proba[i][y_pred[i]] * 100
    status = "Correct" if y_true[i] == y_pred[i] else "Wrong"
    print(f"样本{i+1}: 真实={true_label}, 预测={pred_label}, 置信度={confidence:.1f}% [{status}]")

In [ ]:
# 可视化预测结果
def plot_prediction(dataset, indices, model, class_names, device, n_samples=5):
    """
    可视化模型预测结果。
    Visualize model prediction results.

    Parameters:
    -----------
    dataset : torch Dataset
        测试数据集
    indices : list
        要预测的样本索引
    model : nn.Module
        训练好的模型
    class_names : list
        类别名称列表
    device : torch.device
        计算设备
    n_samples : int
        显示样本数
    """
    model.eval()
    X_list = []
    y_true_list = []
    for idx in indices[:n_samples]:
        img, label = dataset[idx]
        X_list.append(img)
        y_true_list.append(label)

    X_batch = torch.stack(X_list).to(device)
    with torch.no_grad():
        logits = model(X_batch)
        y_proba = torch.softmax(logits, dim=1).cpu().numpy()

    y_true = np.array(y_true_list)

    fig, axes = plt.subplots(2, n_samples, figsize=(12, 5))

    for i in range(n_samples):
        # 显示图像
        axes[0, i].imshow(X_list[i].squeeze(), cmap='gray')
        pred_label = np.argmax(y_proba[i])
        color = 'green' if y_true[i] == pred_label else 'red'
        axes[0, i].set_title(f"True: {class_names[y_true[i]]}\n"
                             f"Pred: {class_names[pred_label]}",
                             color=color, fontsize=9)
        axes[0, i].axis('off')

        # 显示概率分布
        axes[1, i].barh(range(10), y_proba[i])
        axes[1, i].set_yticks(range(10))
        axes[1, i].set_yticklabels(class_names, fontsize=7)
        axes[1, i].set_xlim(0, 1)
        axes[1, i].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# 可视化前5个测试样本的预测
plot_prediction(test_set, list(range(5)), model, class_names, device)

## 10. 混淆矩阵分析

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 对整个测试集进行预测
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        _, predicted = torch.max(logits, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

y_pred_all = np.array(all_preds)
y_test_all = np.array(all_labels)

# 计算混淆矩阵
cm = confusion_matrix(y_test_all, y_pred_all)

# 可视化混淆矩阵
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 打印分类报告
print("分类报告:")
print("=" * 60)
print(classification_report(y_test_all, y_pred_all, target_names=class_names))

## TF vs PyTorch 对照

### 核心API映射表

| 功能 | TensorFlow / Keras | PyTorch |
|------|-------------------|--------|
| **数据加载** | `keras.datasets.fashion_mnist.load_data()` | `torchvision.datasets.FashionMNIST()` |
| **数据格式** | NumPy数组 `(H, W)` | Tensor `(C, H, W)` |
| **归一化** | 手动 `X / 255.0` | `transforms.ToTensor()` 自动归一化 |
| **批量处理** | `model.fit(batch_size=32)` | `DataLoader(batch_size=32)` |
| **构建模型** | `keras.Sequential([Dense(...)])` | `nn.Sequential(nn.Linear(...))` |
| **展平层** | `keras.layers.Flatten()` | `nn.Flatten(start_dim=1)` |
| **全连接层** | `Dense(300, activation='relu')` | `nn.Linear(784, 300)` + `nn.ReLU()` |
| **输出层** | `Dense(10, activation='softmax')` | `nn.Linear(100, 10)` (无softmax) |
| **损失函数** | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss()` |
| **优化器** | `optimizer='sgd'` | `optim.SGD(params, lr=0.01)` |
| **编译** | `model.compile(loss, optimizer, metrics)` | 分别定义 `criterion` 和 `optimizer` |
| **训练** | `model.fit(X, y, epochs=30)` | 手动训练循环 |
| **评估** | `model.evaluate(X, y)` | 手动评估循环 |
| **预测** | `model.predict(X)` → 概率 | `model(X)` → logits，需手动softmax |
| **模型保存** | `model.save('model.h5')` | `torch.save(model.state_dict(), 'model.pt')` |

### 设计哲学差异

1. **Keras**: 高层API，声明式编程，`fit/evaluate/predict`封装了训练细节
2. **PyTorch**: 低层API，命令式编程，训练循环完全可控，调试更方便
3. **Keras**: 激活函数是层的参数 (`Dense(activation='relu')`)
4. **PyTorch**: 激活函数是独立的层 (`nn.Linear()` + `nn.ReLU()`)
5. **Keras**: 输出层包含Softmax，损失函数接收概率
6. **PyTorch**: 输出层不含Softmax，`CrossEntropyLoss`内部处理，数值更稳定

## 小结

### 多分类任务的关键点

1. **数据归一化**: 像素值缩放到[0,1]对训练至关重要
2. **输出层设计**: PyTorch输出原始logits，CrossEntropyLoss内置Softmax
3. **损失函数**: 使用CrossEntropyLoss，接受整数标签，无需one-hot编码
4. **预测输出**: 需手动对logits应用Softmax获取概率分布，使用argmax获取类别

### 模型性能分析

从混淆矩阵可以看出：
- 容易混淆的类别：Shirt vs T-shirt/top, Pullover vs Coat
- 这些类别在视觉上确实相似

### 改进方向

1. 增加网络深度或宽度
2. 添加Dropout正则化（`nn.Dropout(p=0.5)`）
3. 使用更高级的优化器（`optim.Adam`）
4. 使用学习率调度器（`optim.lr_scheduler`）
5. 使用卷积神经网络(CNN)替代MLP

---

## 练习

### 练习1：使用nn.Module自定义模型

将上面的`nn.Sequential`模型改写为继承`nn.Module`的类实现。提示：
- 在`__init__`中定义各层
- 在`forward`中定义前向传播逻辑
- 对比两种方式的优缺点

```python
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 在此定义各层
        pass
    
    def forward(self, x):
        # 在此定义前向传播
        pass
```

### 练习2：添加Dropout和BatchNorm

在隐藏层之间添加`nn.Dropout(p=0.3)`和`nn.BatchNorm1d()`，观察：
- 训练和验证的loss曲线是否更接近（减少过拟合）
- 测试集准确率是否提升
- 注意：`model.eval()`会自动关闭Dropout和BatchNorm

### 练习3：切换优化器和学习率调度

1. 将SGD替换为`optim.Adam(model.parameters(), lr=0.001)`，对比收敛速度
2. 添加学习率调度器：`scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)`
3. 在每个epoch结束后调用`scheduler.step()`，观察学习率变化对训练的影响